In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vivit import ViViTBackbone
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+ViViT + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+2D Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:1' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-08-29 18:39:06,793]::2549926179::DEBUG::=============r2plus1d+ViViT + AST fusion after transformer blocks+2D Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:1


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            final_data_set.extend(pickle.load(output_file))
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
print(1)

1


In [5]:
validation_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/valid_set.dat","valid")
print(1)

1


In [6]:
train_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/train_set.dat","train")
print(1)

1


In [7]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [8]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [9]:
num_workerssz = 20
lr = 3e-05
epochs = 120
print(1)
# Multi-seed setting
SEEDS = [42, 123, 777, 1004, 2024]
patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


1


In [10]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)
print(1)


1


In [11]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [12]:
# ============================================================
# Multi-seed Training
# ============================================================

# Only ONE checkpoint is kept on disk.
# If a better validation MAE is found across any seed, this file is overwritten.
best_model_path = "./save_swintransformer_folder/model_best_multiseed.pth"

seed_results = []
global_best_mae = float("inf")
global_best_seed = None
global_best_epoch = None

with torch.cuda.device(1):

    for seed in SEEDS:

        print("\n" + "=" * 80)
        print(f"Start training | seed = {seed}")
        print("=" * 80)

        # ----------------------------------------------------
        # Reproducibility
        # ----------------------------------------------------
        set_seed(seed)

        generator = torch.Generator()
        generator.manual_seed(seed)

        # Recreate DataLoaders for every seed so that shuffle
        # order is independently controlled.
        train_dataloader = DataLoader(
            dataset=train_set_data,
            batch_size=batchsz,
            shuffle=True,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=generator
        )

        val_dataloader = DataLoader(
            dataset=val_set_data,
            batch_size=batchsz,
            shuffle=False,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=generator
        )

        # ----------------------------------------------------
        # Fresh model / optimizer for each seed
        # ----------------------------------------------------
        model=ViViTBackbone(2, 14, 14, 2, 2, 2, 5, 96, 6, 8, 384, model=3)#num_frames=frame_nums,batchsz = batchsz
        model.to(device)
        criterion = torch.nn.L1Loss().to(device)  # 손실함수
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr) # 옵티마이저

        seed_best_mae = float("inf")
        seed_best_epoch = -1
        no_improve_count = 0

        for i in range(start_epoch, epochs):

            train_avg_loss = 0
            val_avg_loss = 0

            train_avg_o_loss = 0
            train_avg_c_loss = 0
            train_avg_e_loss = 0
            train_avg_a_loss = 0
            train_avg_n_loss = 0

            val_avg_o_loss = 0
            val_avg_c_loss = 0
            val_avg_e_loss = 0
            val_avg_a_loss = 0
            val_avg_n_loss = 0

            # ------------------------------------------------
            # Train
            # ------------------------------------------------
            model.train()

            for audiodata, fullshot, big_five_data in tqdm(
                train_dataloader,
                desc=f"Seed {seed} | Train {i + 1}/{epochs}"
            ):
                fullshot = fullshot.permute(0, 4, 1, 2, 3)
                big_five_data = big_five_data.permute(0, 2, 1)
                big_five_data = big_five_data.squeeze()
                fullshot = fullshot.to(device)
                big_five_data = big_five_data.to(device)
                audiodata = audiodata.to(device)

                optimizer.zero_grad()

                hypothesis = model(fullshot, audiodata)
                loss = criterion(hypothesis, big_five_data)

                loss.backward()
                optimizer.step()

                train_avg_loss += loss

            train_avg_loss = train_avg_loss / len(train_dataloader)

            # ------------------------------------------------
            # Validation
            # ------------------------------------------------
            model.eval()

            with torch.no_grad():

                for audiodata, fullshot, big_five_data in tqdm(
                    val_dataloader,
                    desc=f"Seed {seed} | Valid {i + 1}/{epochs}"
                ):
                    fullshot = fullshot.permute(0, 4, 1, 2, 3)
                    big_five_data = big_five_data.permute(0, 2, 1)
                    big_five_data = big_five_data.squeeze()
                    audiodata = audiodata.to(device)
                    fullshot = fullshot.to(device)
                    big_five_data = big_five_data.to(device)

                    hypothesis = model(fullshot, audiodata)
                    val_loss = criterion(hypothesis, big_five_data)

                    hypothesiso, hypothesisc, hypothesise, hypothesisa, hypothesisn = \
                        extract_ocean(hypothesis, len(fullshot))

                    big_five_datao, big_five_datac, big_five_datae, \
                        big_five_dataa, big_five_datan = \
                        extract_ocean(big_five_data, len(fullshot))

                    val_o_loss = criterion(hypothesiso, big_five_datao)
                    val_c_loss = criterion(hypothesisc, big_five_datac)
                    val_e_loss = criterion(hypothesise, big_five_datae)
                    val_a_loss = criterion(hypothesisa, big_five_dataa)
                    val_n_loss = criterion(hypothesisn, big_five_datan)

                    val_avg_loss += val_loss
                    val_avg_o_loss += val_o_loss
                    val_avg_c_loss += val_c_loss
                    val_avg_e_loss += val_e_loss
                    val_avg_a_loss += val_a_loss
                    val_avg_n_loss += val_n_loss

                val_avg_loss = val_avg_loss / len(val_dataloader)
                val_avg_o_loss = val_avg_o_loss / len(val_dataloader)
                val_avg_c_loss = val_avg_c_loss / len(val_dataloader)
                val_avg_e_loss = val_avg_e_loss / len(val_dataloader)
                val_avg_a_loss = val_avg_a_loss / len(val_dataloader)
                val_avg_n_loss = val_avg_n_loss / len(val_dataloader)

            current_val_mae = val_avg_loss.item()

            # ------------------------------------------------
            # Per-seed best + patience counter
            # ------------------------------------------------
            if current_val_mae < seed_best_mae:
                seed_best_mae = current_val_mae
                seed_best_epoch = i + 1
                no_improve_count = 0
            else:
                no_improve_count += 1

            # ------------------------------------------------
            # Global best checkpoint across ALL seeds.
            # Only this ONE file is kept; it is overwritten.
            # ------------------------------------------------
            if current_val_mae < global_best_mae:
                global_best_mae = current_val_mae
                global_best_seed = seed
                global_best_epoch = i + 1

                torch.save({
                    "seed": seed,
                    "epoch": i + 1,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "loss": current_val_mae,
                }, best_model_path)

                logger.info(
                    "Global best updated | "
                    "seed: {} | epoch: {} | 1-MAE: {:.4f}".format(
                        seed,
                        i + 1,
                        1.0 - current_val_mae
                    )
                )

            logger.debug(
                "Seed: {} | Epoch: {} | 1-MAE: {:.4f} | "
                "1-val_avg_o_loss: {:.4f} | "
                "1-val_avg_c_loss: {:.4f} | "
                "1-val_avg_e_loss: {:.4f} | "
                "1-val_avg_a_loss: {:.4f} | "
                "1-val_avg_n_loss: {:.4f}".format(
                    seed,
                    i + 1,
                    1.0 - current_val_mae,
                    1.0 - val_avg_o_loss.item(),
                    1.0 - val_avg_c_loss.item(),
                    1.0 - val_avg_e_loss.item(),
                    1.0 - val_avg_a_loss.item(),
                    1.0 - val_avg_n_loss.item()
                )
            )

            torch.cuda.empty_cache()

            # ------------------------------------------------
            # Early stopping for current seed
            # ------------------------------------------------
            if no_improve_count >= patience:
                logger.info(
                    "Early stopping | seed: {} | epoch: {} | "
                    "best epoch: {} | best 1-MAE: {:.4f}".format(
                        seed,
                        i + 1,
                        seed_best_epoch,
                        1.0 - seed_best_mae
                    )
                )
                break

        # ----------------------------------------------------
        # Keep scalar results only; do not retain seed models.
        # ----------------------------------------------------
        seed_results.append({
            "seed": seed,
            "best_epoch": seed_best_epoch,
            "best_mae": seed_best_mae,
            "best_1_mae": 1.0 - seed_best_mae
        })

        print(
            f"Seed {seed} finished | "
            f"Best epoch = {seed_best_epoch} | "
            f"Best MAE = {seed_best_mae:.6f} | "
            f"Best 1-MAE = {1.0 - seed_best_mae:.6f}"
        )

        # ----------------------------------------------------
        # Release memory before next seed
        # ----------------------------------------------------
        del model
        del optimizer
        del criterion
        del train_dataloader
        del val_dataloader
        del generator

        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# Multi-seed Summary
# ============================================================

all_1_mae = np.array(
    [result["best_1_mae"] for result in seed_results],
    dtype=np.float64
)

mean_1_mae = all_1_mae.mean()
std_1_mae = all_1_mae.std(ddof=1) if len(all_1_mae) > 1 else 0.0

print("\n" + "=" * 80)
print("MULTI-SEED RESULTS")
print("=" * 80)

for result in seed_results:
    print(
        "Seed {:>4} | Best Epoch {:>3} | "
        "MAE {:.6f} | 1-MAE {:.6f}".format(
            result["seed"],
            result["best_epoch"],
            result["best_mae"],
            result["best_1_mae"]
        )
    )

print("-" * 80)
print(f"Mean 1-MAE : {mean_1_mae:.6f}")
print(f"Std  1-MAE : {std_1_mae:.6f}")
print(
    f"Global best: seed={global_best_seed}, "
    f"epoch={global_best_epoch}, "
    f"1-MAE={1.0 - global_best_mae:.6f}"
)
print(f"Saved checkpoint: {best_model_path}")
print("=" * 80)



Start training | seed = 42
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 42 | Train 1/120: 100%|█████████▉| 1497/1498 [06:29<00:00,  3.92it/s]/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/torch/nn/modules/loss.py:101: UserWarning: Using a target size (torch.Size([5])) that is different to the input size (torch.Size([1, 5])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
Seed 42 | Train 1/120: 100%|██████████| 1498/1498 [06:30<00:00,  3.83it/s]
Seed 42 | Valid 1/120: 100%|██████████| 498/498 [00:49<00:00, 10.04it/s]
[2026-08-29 18:46:39,111]::4131322566::INFO::Global best updated | seed: 42 | epoch: 1 | 1-MAE: 0.8985
[2026-08-29 18:46:39,113]::4131322566::DEBUG::Seed: 42 | Epoch: 1 | 1-MAE: 0.8985 | 1-val_avg_o_loss: 0.9082 | 1-val_avg_c_loss: 0.9027 | 1-val_avg_e_loss: 0.9021 | 1-val_avg_a_loss: 0.8797 | 1-val_avg_n_loss: 0.9000
Seed 42 | Train 2/120: 100%|██████████| 1498/1498 [06:35<00:00,  3.79it/s]
Seed 42 | Valid 2/120: 1

Seed 42 finished | Best epoch = 9 | Best MAE = 0.084803 | Best 1-MAE = 0.915197

Start training | seed = 123
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 123 | Train 1/120: 100%|██████████| 1498/1498 [06:35<00:00,  3.78it/s]
Seed 123 | Valid 1/120: 100%|██████████| 498/498 [00:50<00:00,  9.82it/s]
[2026-08-29 22:23:37,635]::4131322566::DEBUG::Seed: 123 | Epoch: 1 | 1-MAE: 0.8990 | 1-val_avg_o_loss: 0.8997 | 1-val_avg_c_loss: 0.8966 | 1-val_avg_e_loss: 0.9064 | 1-val_avg_a_loss: 0.8972 | 1-val_avg_n_loss: 0.8951
Seed 123 | Train 2/120: 100%|██████████| 1498/1498 [06:36<00:00,  3.78it/s]
Seed 123 | Valid 2/120: 100%|██████████| 498/498 [00:50<00:00,  9.85it/s]
[2026-08-29 22:31:06,579]::4131322566::DEBUG::Seed: 123 | Epoch: 2 | 1-MAE: 0.9007 | 1-val_avg_o_loss: 0.8849 | 1-val_avg_c_loss: 0.9093 | 1-val_avg_e_loss: 0.9010 | 1-val_avg_a_loss: 0.8974 | 1-val_avg_n_loss: 0.9108
Seed 123 | Train 3/120: 100%|██████████| 1498/1498 [06:36<00:00,  3.77it/s]
Seed 123 | Valid 3/120: 100%|██████████| 498/498 [00:50<00:00,  9.88it/s]
[2026-08-29 22:38:36,368]::4131322566::DEBUG::Seed: 123 | Epoch: 3 | 1-MAE: 0.9109 | 1-val_avg_o_loss: 0.9120 | 1-

Seed 123 finished | Best epoch = 21 | Best MAE = 0.084653 | Best 1-MAE = 0.915347

Start training | seed = 777
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


...

Seed 777 finished | Best epoch = 7 | Best MAE = 0.084431 | Best 1-MAE = 0.915569

Start training | seed = 1004
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 1004 | Train 1/120: 100%|██████████| 1498/1498 [06:37<00:00,  3.77it/s]
Seed 1004 | Valid 1/120: 100%|██████████| 498/498 [00:50<00:00,  9.81it/s]
[2026-08-30 06:53:37,704]::4131322566::DEBUG::Seed: 1004 | Epoch: 1 | 1-MAE: 0.9050 | 1-val_avg_o_loss: 0.9036 | 1-val_avg_c_loss: 0.9054 | 1-val_avg_e_loss: 0.9083 | 1-val_avg_a_loss: 0.8977 | 1-val_avg_n_loss: 0.9100
Seed 1004 | Train 2/120: 100%|██████████| 1498/1498 [06:36<00:00,  3.78it/s]
Seed 1004 | Valid 2/120: 100%|██████████| 498/498 [00:50<00:00,  9.85it/s]
[2026-08-30 07:01:07,046]::4131322566::DEBUG::Seed: 1004 | Epoch: 2 | 1-MAE: 0.9115 | 1-val_avg_o_loss: 0.9118 | 1-val_avg_c_loss: 0.9107 | 1-val_avg_e_loss: 0.9134 | 1-val_avg_a_loss: 0.9090 | 1-val_avg_n_loss: 0.9127
Seed 1004 | Train 3/120: 100%|██████████| 1498/1498 [06:36<00:00,  3.78it/s]
Seed 1004 | Valid 3/120: 100%|██████████| 498/498 [00:50<00:00,  9.90it/s]
[2026-08-30 07:08:36,029]::4131322566::DEBUG::Seed: 1004 | Epoch: 3 | 1-MAE: 0.9115 | 1-val_avg_o_loss: 0.

Seed 1004 finished | Best epoch = 6 | Best MAE = 0.084170 | Best 1-MAE = 0.915830

Start training | seed = 2024
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 2024 | Train 1/120: 100%|██████████| 1498/1498 [06:37<00:00,  3.77it/s]
Seed 2024 | Valid 1/120: 100%|██████████| 498/498 [00:50<00:00,  9.79it/s]
[2026-08-30 10:08:43,130]::4131322566::DEBUG::Seed: 2024 | Epoch: 1 | 1-MAE: 0.9048 | 1-val_avg_o_loss: 0.9005 | 1-val_avg_c_loss: 0.9054 | 1-val_avg_e_loss: 0.9118 | 1-val_avg_a_loss: 0.9007 | 1-val_avg_n_loss: 0.9055
Seed 2024 | Train 2/120: 100%|██████████| 1498/1498 [06:37<00:00,  3.77it/s]
Seed 2024 | Valid 2/120: 100%|██████████| 498/498 [00:50<00:00,  9.81it/s]
[2026-08-30 10:16:12,989]::4131322566::DEBUG::Seed: 2024 | Epoch: 2 | 1-MAE: 0.9063 | 1-val_avg_o_loss: 0.9126 | 1-val_avg_c_loss: 0.9066 | 1-val_avg_e_loss: 0.9026 | 1-val_avg_a_loss: 0.9064 | 1-val_avg_n_loss: 0.9034
Seed 2024 | Train 3/120: 100%|██████████| 1498/1498 [06:37<00:00,  3.77it/s]
Seed 2024 | Valid 3/120: 100%|██████████| 498/498 [00:50<00:00,  9.80it/s]
[2026-08-30 10:23:42,957]::4131322566::DEBUG::Seed: 2024 | Epoch: 3 | 1-MAE: 0.9130 | 1-val_avg_o_loss: 0.

Seed 2024 finished | Best epoch = 11 | Best MAE = 0.084718 | Best 1-MAE = 0.915282

MULTI-SEED RESULTS
Seed   42 | Best Epoch   9 | MAE 0.084803 | 1-MAE 0.915197
Seed  123 | Best Epoch  21 | MAE 0.084653 | 1-MAE 0.915347
Seed  777 | Best Epoch   7 | MAE 0.084431 | 1-MAE 0.915569
Seed 1004 | Best Epoch   6 | MAE 0.084170 | 1-MAE 0.915830
Seed 2024 | Best Epoch  11 | MAE 0.084718 | 1-MAE 0.915282
--------------------------------------------------------------------------------
Mean 1-MAE : 0.915445
Std  1-MAE : 0.000256
Global best: seed=1004, epoch=6, 1-MAE=0.915830
Saved checkpoint: ./save_swintransformer_folder/model_best_multiseed.pth
